## SRP340148

**paper:** [PMID: 25490151](https://pmc.ncbi.nlm.nih.gov/articles/PMC4354015/) - Highly Efficient Ligation of Small RNA Molecules for MicroRNA Quantitation by High-Throughput Sequencing, 2014

**date, curator:** 2026-07-10, Sara Carsanaro

**notes**
* rejected miRNA samples because they used PAGE
* rejected SRP151074, [PMID: 30086302](https://pmc.ncbi.nlm.nih.gov/articles/PMC6114144/). the only libraries that would qualify as healthy wild type are annotated as part of this experiment
* from methods:
    * Illumina Truseq Total RNA library Prep Kit
    * age: 5-day-old flies
    * sex is all M because testis

### annotation summary

In [ ]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

In [ ]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

### set variables, import packages, define functions

In [1]:
experiment_id = "SRP340148"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,,,,,,,7240,,,,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,,,,,,,7240,,,,,,Dsim_testis_RNAseq_replicate1,"SAMN33260509,GSM7039340",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA librari

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['testis']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000473'
library.loc[:,'anatName'] = 'testis'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,,,,,7240,,,,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,,,,,7240,,,,,,Dsim_testis_RNAseq_replicate1,"SAMN33260509,GSM7039340",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['5 day old adults']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,perfect match,,,,7240,,,,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,perfect match,,,,7240,,,,,,Dsim_testis_RNAseq_replicate1,"SAMN33260509,GSM7039340",,,,,,,,,,,10/07/2026,"For 

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [9]:
library.loc[:,'sex'] = 'M'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,,,,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,,,,,,Dsim_testis_RNAseq_replicate1,"SAMN33260509,GSM7039340",,,,,,,,,,,10/07/2026,"Fo

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded Total RNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'ribo-minus'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",,,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,r

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [12]:
library.loc[:,'sampleAge_value'] = '5'
library.loc[:,'sampleAge_unit'] = 'day post eclosion'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",5,day post eclosion,,,,,,,,,10/07/2026,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [13]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-14'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",5,day post eclosion,,,,,,,,SAC,2026-07-14,"For small RNA analysis, we extracted RNA from testes and accessory glands of 7-day-old Dsim w[XD1] strain using Trizol (Invitrogen). 1 μg of total RNA was used to prepare small RNA libraries as described3, with the addition of QIAseq miRNA Library QC Spike-ins for normalization (Qiagen). Adenylation of 3' linker was performed in a 40 µL reaction at 65°C for 1 hr containing 200 pmol 3' linker, 1X 5' DNA adenylation reaction buffer, 100 nM ATP and 200 pmol Mth RNA ligase and the reaction is terminated by heated to 85°C for 5 min. Adenylated 3' linker was then precipitated using ethanol and was used for 3' ligation reaction containing 10% PEG8000, 1X RNA ligase buffer, 20 µM adenylated 3' linker and 100 U T4 RNA Ligase 2 truncated K227Q. The 3' ligation reaction was performed at 4°C overnight and the products were purified using 15% Urea-PAGE gel. The small RNA-3' linker hybrid was then subjected to 5' ligation reaction at 37°C for 4 hr containing 20% PEG8000, 1X RNA ligase buffer, 1 mM ATP, 10 µM RNA oligo, 20 U RNaseOUT and 5 U T4 RNA ligase 1. cDNA synthesis reaction was then proceeded immediately by adding following components to the ligated product: 2 μl 5x RT buffer, 0.75 μl 100 mM DTT, 1 μl 1 μM Illumina RT Primer, and 0.5 μl 10 mM dNTPs. The RT mix was incubated at 65 C for 5 min and cooled to room temperature and transfer on to ice. 0.5 µL of superscript III RT enzyme and 0.5 µL RNase OUT were added to the RT mix and the reaction was carried out at 50°C for 1 hr. cDNA libraries were amplified using 15 cycles of PCR with forward and illumine index reverse primers and the amplified libraries were purified by 8% non-denaturing acrylamide gel. Purified libraries were sequenced on HiSeq2500 using SR50 at the New York Genome Center. We used Dsim w[XD1], which were used for PacBio genome sequencing (Chakraborty et al. Genome Research 2021). We isolated total RNA from ~ 5-day-old flies and for Dsim samples. We extracted RNA from testes (dissected free of accessory glands) using Trizol (Invitrogen). We made two independent dissections to generate biologically replicate RNA samples, whose quality was assessed by Bioanalyzer. We used the Illumina TruSeq Total RNA library Prep Kit LT to make RNA-seq libraries from 650 ng of total RNA. Manufacturer's protocol was followed except for using 8 cycles of PCR to amplify the final library instead of the recommended 15 cycles, to minimize artifacts caused by PCR amplification. All samples were pooled together, using the barcoded adapters provided by the manufacturer, over 2 flow cells of a HiSeq2500 and sequenced using PE75 at the New York Genome Center. Single-end sequencing for small RNA sequencing and paired-end sequencing for RNA sequencing data",,,,D. simulans testis,,,,TRANSCRIPTOMIC,cDNA
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded To

#### comments

In [14]:
library.loc[:,'comment'] = 'PMID: 34862477'

#### save complete file with correct columns

In [15]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039341,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
1,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7039340,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate1,"SAMN33260509,GSM7039340",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
2,SRX12495357,SRP340148,Illumina HiSeq 2500,SRS10458306,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5611561,testis,5 day old adults,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsech_testis_RNAseq_replicate2,"SAMN22069417,GSM5611561",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
3,SRX12495356,SRP340148,Illumina HiSeq 2500,SRS10458305,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5611560,testis,5 day old adults,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsech_testis_RNAseq_replicate1,"SAMN22069418,GSM5611560",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
4,SRX12495355,SRP340148,Illumina HiSeq 2500,SRS10458304,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5611559,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7226,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dmau_testis_RNAseq_replicate2,"SAMN22069419,GSM5611559",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
5,SRX12495354,SRP340148,Illumina HiSeq 2500,SRS10458303,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5611558,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7226,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dmau_testis_RNAseq_replicate1,"SAMN22069420,GSM5611558",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14


### experiment annotations

In [16]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP340148,Rapid evolutionary dynamics of an expanding family of Drosophila meiotic drive factors and their hpRNA suppressors,"The Dox meiotic drive system distorts the sex-ratio in Drosophila simulans. Here, the authors reconstruct the step-wise emergence, and recent amplification of Dox superfamily genes in parallel with the emergence of autosomal hairpin RNA-class siRNA loci that target subsets of these putative drivers Overall design: The study reports small RNA and RNA-seq data from testis from 3 closely related species in the Drosophila simulans clade.",SRA,,,,QIAseq miRNA Library Kit,full_length,GSE185361,PRJNA768815,34862477,,10.1038/s41559-021-01592-z,,


#### experiment and protocol details

In [17]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [18]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP340148,Rapid evolutionary dynamics of an expanding family of Drosophila meiotic drive factors and their hpRNA suppressors,"The Dox meiotic drive system distorts the sex-ratio in Drosophila simulans. Here, the authors reconstruct the step-wise emergence, and recent amplification of Dox superfamily genes in parallel with the emergence of autosomal hairpin RNA-class siRNA loci that target subsets of these putative drivers Overall design: The study reports small RNA and RNA-seq data from testis from 3 closely related species in the Drosophila simulans clade.",SRA,partial,Bgee 1K,6,TruSeq Stranded Total RNA,full_length,GSE185361,PRJNA768815,34862477,,10.1038/s41559-021-01592-z,,


#### paper and xrefs

In [19]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '34862477'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8665063/'
experiment.loc[:,'DOI'] = '10.1038/s41559-021-01592-z'
experiment.loc[:,'xrefs'] = '30086302[PMID]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP340148,Rapid evolutionary dynamics of an expanding family of Drosophila meiotic drive factors and their hpRNA suppressors,"The Dox meiotic drive system distorts the sex-ratio in Drosophila simulans. Here, the authors reconstruct the step-wise emergence, and recent amplification of Dox superfamily genes in parallel with the emergence of autosomal hairpin RNA-class siRNA loci that target subsets of these putative drivers Overall design: The study reports small RNA and RNA-seq data from testis from 3 closely related species in the Drosophila simulans clade.",SRA,partial,Bgee 1K,6,TruSeq Stranded Total RNA,full_length,GSE185361,PRJNA768815,34862477,https://pmc.ncbi.nlm.nih.gov/articles/PMC8665063/,10.1038/s41559-021-01592-z,30086302[PMID],


#### comments

In [20]:
experiment.loc[:,'comment'] = 'removed miRNA seq libraries with PAGE'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP340148,Rapid evolutionary dynamics of an expanding family of Drosophila meiotic drive factors and their hpRNA suppressors,"The Dox meiotic drive system distorts the sex-ratio in Drosophila simulans. Here, the authors reconstruct the step-wise emergence, and recent amplification of Dox superfamily genes in parallel with the emergence of autosomal hairpin RNA-class siRNA loci that target subsets of these putative drivers Overall design: The study reports small RNA and RNA-seq data from testis from 3 closely related species in the Drosophila simulans clade.",SRA,partial,Bgee 1K,6,TruSeq Stranded Total RNA,full_length,GSE185361,PRJNA768815,34862477,https://pmc.ncbi.nlm.nih.gov/articles/PMC8665063/,10.1038/s41559-021-01592-z,30086302[PMID],removed miRNA seq libraries with PAGE


#### save complete file

In [21]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [23]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 6
Errors: 6
Warnings: 0
Top codes:
  - BAD_VALUE: 6


#### check columns match

In [25]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [26]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
67861,SRX22220016,SRP468365,Illumina NovaSeq 6000,SRS19275089,UBERON:0001495,pectoral muscle,UBERON:0000113,post-juvenile adult stage,,pectoralis,adult,perfect match,partial sampling,other,M,NA,,59729,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,ZF3_13_1_pectoralis,SAMN37982571,,,,,,,"PMID:41747044, mRNA was enriched from 400 ng t...",,,ANN,2026-07-14
67862,SRX22220015,SRP468365,Illumina NovaSeq 6000,SRS19275088,UBERON:0001495,pectoral muscle,UBERON:0000113,post-juvenile adult stage,,pectoralis,adult,perfect match,partial sampling,other,M,NA,,59729,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,ZF2_13_1_pectoralis,SAMN37982570,,,,,,,"PMID:41747044, mRNA was enriched from 400 ng t...",,,ANN,2026-07-14
67863,SRX19338861,SRP340148,Illumina HiSeq 2500,SRS16735467,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate2,"SAMN33260508,GSM7039341",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
67864,SRX19338860,SRP340148,Illumina HiSeq 2500,SRS16735466,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7240,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsim_testis_RNAseq_replicate1,"SAMN33260509,GSM7039340",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
67865,SRX12495357,SRP340148,Illumina HiSeq 2500,SRS10458306,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,5 day old adults,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsech_testis_RNAseq_replicate2,"SAMN22069417,GSM5611561",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
67866,SRX12495356,SRP340148,Illumina HiSeq 2500,SRS10458305,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,5 day old adults,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dsech_testis_RNAseq_replicate1,"SAMN22069418,GSM5611560",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14
67867,SRX12495355,SRP340148,Illumina HiSeq 2500,SRS10458304,UBERON:0000473,testis,UBERON:0000066,fully formed stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,5 day old adults,perfect match,not documented,perfect match,M,,,7226,TruSeq Stranded Total RNA,full_length,ribo-minus,,,Dmau_testis_RNAseq_replicate2,"SAMN22069419,GSM5611559",5,day post eclosion,,,,,PMID: 34862477,,,SAC,2026-07-14


In [27]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1286,DRP005429,Construction of complete Tupaia belangeri tran...,The comprehensive transcriptome database of Tu...,SRA,partial,Bgee 1K,9,"TruSeq Stranded Total RNA, TruSeq Stranded mRNA",full_length,,PRJDB7591,31451757,https://pmc.ncbi.nlm.nih.gov/articles/PMC6710255/,10.1038/s41598-019-48867-x,,
1287,SRP468365,Tissue-specific RNAseq of Taeniopygia guttata ...,RNA sequencing reads of the tissue samples fro...,SRA,total,Bgee 1K,35,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1032078,41747044,https://www.biorxiv.org/content/10.1101/2024.0...,10.1126/science.adt1522,,
1288,SRP340148,Rapid evolutionary dynamics of an expanding fa...,The Dox meiotic drive system distorts the sex-...,SRA,partial,Bgee 1K,6,TruSeq Stranded Total RNA,full_length,GSE185361,PRJNA768815,34862477,https://pmc.ncbi.nlm.nih.gov/articles/PMC8665063/,10.1038/s41559-021-01592-z,30086302[PMID],removed miRNA seq libraries with PAGE


### add annotations to git

In [29]:
! git pull

Already up to date.


In [30]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 296e0a9] adding annotated bulk experiment SRP340148
 2 files changed, 7 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.57 KiB | 2.57 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   0c3e717..296e0a9  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push